## 1. Imports & Load Raw Data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os.path as path

In [3]:
df = pd.read_csv('dataset/forestfires.csv')
y_target = 'area'
df.head(5)

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0


## 2. Handle Missing / Invalid Values

In [4]:
missing_feature = df.columns[df.isnull().any()].to_list()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 0
Duplicates: 4
Missing Feature:
[]


## 3. Handling Duplicated

In [5]:
df = df.drop_duplicates(keep='last')

jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah data duplikat: {jumlah_duplikat}")

Jumlah data duplikat: 0


## 4. Feature Engineering

### 4.1 Handling Feature Target

In [6]:
df[y_target] = np.log1p(df[y_target])

### 4.2 Handling Feature Month

In [7]:
df['is_summer'] = df['month'].isin(['jun', 'jul', 'aug', 'sep']).astype(int)

### 4.3 Handling Feature numeric

In [8]:
df['temp_to_RH'] = df['temp'] / (df['RH'] + 1e-5) # 1e-5 untuk menghindari pembagian dengan nol
df['fire_spread_potential'] = df['wind'] * df['ISI']
df['distance_from_center'] = np.sqrt(df['X']**2 + df['Y']**2)

In [9]:
df.head(2)

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area,is_summer,temp_to_RH,fire_spread_potential,distance_from_center
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0,0,0.160784,34.17,8.602325
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0,0,0.545454,6.03,8.062258


## 5. Analisis & Handling Outliers

In [10]:
feature_numerik = df.select_dtypes(include=np.number).columns
feature_numerik = [col for col in feature_numerik if col != 'is_summer']

Q1 = df[feature_numerik].quantile(0.25)
Q3 = df[feature_numerik].quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sebelum Dihapus: {outliers.shape[0]}")

#delete outliers
df = df.loc[((df[feature_numerik] >= lower_bound) & (df[feature_numerik] <= upper_bound)).all(axis=1)]
outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sesudah Dihapus: {outliers.shape[0]}")

Jumlah Outlier Sebelum Dihapus: 176
Jumlah Outlier Sesudah Dihapus: 0


## 6.Save Cleaned Dataset

In [11]:
file_path = 'dataset/forestfires_CLEANING.csv'

if not path.exists(file_path):
    df.to_csv(file_path, index=False)
    print('File belum ada. Berhasil menyimpan dataset baru!')
else:
    # Jika SUDAH ADA, lewati proses penyimpanan
    print('File sudah ada. Proses penyimpanan CSV dilewati (skip)')

df.head()

File sudah ada. Proses penyimpanan CSV dilewati (skip)


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area,is_summer,temp_to_RH,fire_spread_potential,distance_from_center
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0,0,0.160784,34.17,8.602325
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0,0,0.545454,6.03,8.062258
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0,0,0.442424,8.71,8.062258
5,8,6,aug,sun,92.3,85.3,488.0,14.7,22.2,29,5.4,0.0,0.0,1,0.765517,79.38,10.000000
6,8,6,aug,mon,92.3,88.9,495.6,8.5,24.1,27,3.1,0.0,0.0,1,0.892592,26.35,10.000000
